# 🔍 Exploratory Data Analysis (EDA)
## Security Feature Engineering Pipeline

**Assignment**: ARZENS Track 09 – AI, Automation & Security Engineering (Advanced)  
**Author**: Abdul Rehman  
**Date**: July 2026

---

This notebook performs comprehensive exploratory data analysis on:
1. Raw security events (`sample_raw_events.jsonl`)
2. Engineered ML-ready features (`sample_features.csv`)

**Sections:**
- Data Loading & Overview
- Event Distribution Analysis
- Temporal Distribution
- Feature Statistics & Distributions
- Correlation Analysis
- Outlier Detection
- Privacy Impact Assessment
- Key Findings & Recommendations

In [ ]:
# === Imports ===
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Libraries loaded successfully!')

## 1. Data Loading & Overview

In [ ]:
# === Load Raw Events ===
raw_events = []
with open('Task02_Security_Feature_Extractor/sample_raw_events.jsonl', 'r') as f:
    for line in f:
        raw_events.append(json.loads(line.strip()))

events_df = pd.DataFrame(raw_events)
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'])

print(f'Total raw events loaded: {len(events_df)}')
print(f'Time range: {events_df["timestamp"].min()} to {events_df["timestamp"].max()}')
print(f'Unique source IPs: {events_df["source_ip"].nunique()}')
print(f'Unique users: {events_df["user_id"].dropna().nunique()}')
print(f'\nEvent types:')
print(events_df['event_type'].value_counts())
print(f'\nDataFrame shape: {events_df.shape}')
events_df.head()

In [ ]:
# === Load Engineered Features ===
features_df = pd.read_csv('Task02_Security_Feature_Extractor/sample_features.csv')
features_df['window_start'] = pd.to_datetime(features_df['window_start'])

print(f'Total feature records: {len(features_df)}')
print(f'Number of columns: {len(features_df.columns)}')
print(f'Columns: {list(features_df.columns)}')
print(f'\nUnique entities: {features_df["entity_id"].nunique()}')
print(f'Window range: {features_df["window_start"].min()} to {features_df["window_start"].max()}')
features_df.head()

In [ ]:
# === Feature Summary Statistics ===
numeric_cols = features_df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')
print()
features_df[numeric_cols].describe().round(4)

## 2. Event Distribution Analysis

In [ ]:
# === Event Type Distribution ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
event_counts = events_df['event_type'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
event_counts.plot(kind='bar', ax=axes[0], color=colors[:len(event_counts)], edgecolor='white', linewidth=1.5)
axes[0].set_title('Event Count by Type', fontweight='bold')
axes[0].set_xlabel('Event Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, (idx, val) in enumerate(event_counts.items()):
    axes[0].text(i, val + 2, str(val), ha='center', fontweight='bold')

# Pie chart
event_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=colors[:len(event_counts)],
                  startangle=90, pctdistance=0.85, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Event Type Proportions', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Event distribution saved to event_distribution.png')

In [ ]:
# === Source IP Activity ===
ip_counts = events_df['source_ip'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 5))
ip_counts.plot(kind='barh', ax=ax, color=sns.color_palette('coolwarm', len(ip_counts)), edgecolor='white')
ax.set_title('Top 15 Source IPs by Event Count', fontweight='bold')
ax.set_xlabel('Number of Events')
ax.set_ylabel('Source IP')
ax.invert_yaxis()
for i, val in enumerate(ip_counts.values):
    ax.text(val + 0.5, i, str(val), va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('source_ip_activity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Temporal Distribution

In [ ]:
# === Events Over Time ===
events_df['hour'] = events_df['timestamp'].dt.hour
events_df['date'] = events_df['timestamp'].dt.date

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hourly distribution
hourly = events_df.groupby('hour').size()
axes[0].bar(hourly.index, hourly.values, color='#3498db', edgecolor='white', alpha=0.9)
axes[0].axvspan(0, 8, alpha=0.1, color='red', label='Off-hours (0-8)')
axes[0].axvspan(18, 24, alpha=0.1, color='red', label='Off-hours (18-24)')
axes[0].set_title('Event Distribution by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Event Count')
axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))

# Daily distribution
daily = events_df.groupby('date').size()
axes[1].bar(range(len(daily)), daily.values, color='#2ecc71', edgecolor='white', alpha=0.9)
axes[1].set_title('Event Distribution by Date', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Event Count')
axes[1].set_xticks(range(len(daily)))
axes[1].set_xticklabels([str(d) for d in daily.index], rotation=15)

plt.tight_layout()
plt.savefig('temporal_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Event Type by Hour Heatmap ===
pivot = events_df.groupby(['hour', 'event_type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot.T, cmap='YlOrRd', annot=True, fmt='d', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Event Count'})
ax.set_title('Event Type × Hour of Day Heatmap', fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Event Type')

plt.tight_layout()
plt.savefig('event_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distributions

In [ ]:
# === Histograms for All Numeric Features ===
feature_cols = [c for c in numeric_cols if c != 'event_count']

n_features = len(feature_cols)
n_cols_plot = 3
n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(18, 4 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    data = features_df[col].dropna()
    axes[i].hist(data, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.3f}')
    axes[i].axvline(data.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.3f}')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Frequency')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions with Mean & Median', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Box Plots for Outlier Visualization ===
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(18, 4 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    data = features_df[col].dropna()
    bp = axes[i].boxplot(data, vert=True, patch_artist=True,
                         boxprops=dict(facecolor='#3498db', alpha=0.7),
                         medianprops=dict(color='red', linewidth=2),
                         flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    
    # Calculate outlier count using IQR method
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)).sum()
    axes[i].set_xlabel(f'Outliers: {outliers} ({outliers/len(data)*100:.1f}%)', fontsize=9, color='red')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Box Plots (IQR Outlier Detection)', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
# === Feature Correlation Heatmap ===
corr_matrix = features_df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (Lower Triangle)', fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Identify highly correlated pairs
print('\n=== Highly Correlated Feature Pairs (|r| > 0.5) ===')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.5:
            print(f'  {corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: r = {corr_matrix.iloc[i, j]:.3f}')

## 6. Outlier Detection

In [ ]:
# === IQR-Based Outlier Analysis ===
print('=== Outlier Detection Summary (IQR Method) ===')
print(f'{"Feature":<35} {"Outliers":>8} {"Percentage":>10} {"Min":>12} {"Max":>12}')
print('=' * 80)

outlier_summary = []
for col in feature_cols:
    data = features_df[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    pct = len(outliers) / len(data) * 100
    print(f'{col:<35} {len(outliers):>8} {pct:>9.1f}% {data.min():>12.4f} {data.max():>12.4f}')
    outlier_summary.append({
        'feature': col, 'outlier_count': len(outliers),
        'outlier_pct': pct, 'lower_bound': lower_bound, 'upper_bound': upper_bound
    })

outlier_df = pd.DataFrame(outlier_summary)
print(f'\nTotal features with outliers: {(outlier_df["outlier_count"] > 0).sum()} / {len(outlier_df)}')

In [ ]:
# === Anomalous Records Identification ===
print('=== Top Anomalous Records ===')
print()

# High geo-velocity (impossible travel)
high_geo = features_df[features_df['geo_velocity'] > 800].sort_values('geo_velocity', ascending=False).head(10)
print(f'Records with geo_velocity > 800 km/h (impossible travel): {len(high_geo)}')
if len(high_geo) > 0:
    print(high_geo[['entity_id', 'window_start', 'geo_velocity']].to_string())
print()

# High failed login ratio (brute-force indicators)
high_flr = features_df[features_df['failed_login_ratio'] > 0.5].sort_values('failed_login_ratio', ascending=False).head(10)
print(f'Records with failed_login_ratio > 0.5 (brute-force): {len(high_flr)}')
if len(high_flr) > 0:
    print(high_flr[['entity_id', 'window_start', 'failed_login_ratio', 'login_frequency']].to_string())
print()

# High DNS entropy (potential DGA/tunneling)
high_dns = features_df[features_df['dns_query_entropy'] > 3.5].sort_values('dns_query_entropy', ascending=False).head(10)
print(f'Records with dns_query_entropy > 3.5 (DGA/tunneling): {len(high_dns)}')
if len(high_dns) > 0:
    print(high_dns[['entity_id', 'window_start', 'dns_query_entropy']].to_string())

## 7. Privacy Impact Assessment

In [ ]:
# === Privacy Controls Analysis ===
print('=== Privacy Impact Assessment ===')
print()

# Check pseudonymization
sample_ids = features_df['entity_id'].unique()[:5]
print('1. PSEUDONYMIZATION CHECK')
print(f'   Entity IDs are SHA-256 hashed: {all(len(str(eid)) == 64 for eid in sample_ids)}')
print(f'   Sample entity IDs:')
for eid in sample_ids:
    print(f'     {eid}')
print()

# Check IP generalization
print('2. IP GENERALIZATION CHECK')
if 'generalized_ip' in features_df.columns:
    sample_ips = features_df['generalized_ip'].dropna().unique()[:5]
    print(f'   Generalized IPs (last octet zeroed):')
    for ip in sample_ips:
        print(f'     {ip}')
    generalized_count = features_df['generalized_ip'].notna().sum()
    print(f'   Records with generalized IPs: {generalized_count} / {len(features_df)}')
else:
    print('   No generalized_ip column found')
print()

# Check differential privacy (noise in failed_login_ratio)
print('3. DIFFERENTIAL PRIVACY CHECK')
flr = features_df['failed_login_ratio']
negative_values = (flr < 0).sum()
above_one = (flr > 1).sum()
print(f'   Failed login ratio values < 0 (noise-induced): {negative_values}')
print(f'   Failed login ratio values > 1 (noise-induced): {above_one}')
print(f'   Evidence of Laplace noise: {"YES" if (negative_values > 0 or above_one > 0) else "NO"}')
print(f'   Range: [{flr.min():.4f}, {flr.max():.4f}]')

## 8. Missing Value Analysis

In [ ]:
# === Missing Values ===
missing = features_df.isnull().sum()
missing_pct = (missing / len(features_df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_report) > 0:
    print('=== Columns with Missing Values ===')
    print(missing_report.to_string())
    
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_report['Missing %'].plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_title('Missing Value Percentage by Column', fontweight='bold')
    ax.set_xlabel('Missing %')
    plt.tight_layout()
    plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing values detected in any column!')
    
# Overall completeness
total_cells = features_df.shape[0] * features_df.shape[1]
total_missing = features_df.isnull().sum().sum()
completeness = (1 - total_missing / total_cells) * 100
print(f'\nOverall data completeness: {completeness:.2f}%')

## 9. Key Findings & Recommendations

In [ ]:
print('=' * 70)
print('              KEY FINDINGS & RECOMMENDATIONS')
print('=' * 70)
print()

print('📊 DATA OVERVIEW:')
print(f'  • {len(events_df)} raw security events across {events_df["event_type"].nunique()} event types')
print(f'  • {len(features_df)} engineered feature records with {len(feature_cols)} features')
print(f'  • {features_df["entity_id"].nunique()} unique entities tracked')
print()

print('🔍 THREAT INDICATORS DETECTED:')
impossible_travel = (features_df['geo_velocity'] > 800).sum()
brute_force = (features_df['failed_login_ratio'] > 0.5).sum()
dns_anomaly = (features_df['dns_query_entropy'] > 3.5).sum()
high_exfil = (features_df['data_exfil_ratio'] > 10).sum()
off_hours = (features_df['off_hours_ratio'] > 0.8).sum()
priv_esc = (features_df['privilege_escalation_score'] > 0).sum()

print(f'  • Impossible travel events (>800 km/h): {impossible_travel}')
print(f'  • Brute-force indicators (FLR > 0.5): {brute_force}')
print(f'  • DNS anomalies (entropy > 3.5): {dns_anomaly}')
print(f'  • Data exfiltration anomalies (ratio > 10): {high_exfil}')
print(f'  • Off-hours heavy access (ratio > 0.8): {off_hours}')
print(f'  • Privilege escalation events: {priv_esc}')
print()

print('🔒 PRIVACY CONTROLS:')
print(f'  • Entity IDs: Pseudonymized via SHA-256 hashing')
print(f'  • IP Addresses: Generalized to /24 subnet')
print(f'  • Differential Privacy: Laplace noise applied to sensitive features')
print()

print('📋 RECOMMENDATIONS:')
print('  1. Investigate impossible travel events — potential credential compromise')
print('  2. Review brute-force indicators and implement rate limiting')
print('  3. Analyze high DNS entropy queries for DGA/C2 activity')
print('  4. Monitor data exfiltration ratios for insider threat patterns')
print('  5. Correlate off-hours access with privilege escalation events')
print('  6. Consider tuning differential privacy epsilon for feature utility')
print()
print('=' * 70)

In [ ]:
print('\n✅ EDA Complete! All visualizations have been saved.')
print('\nGenerated files:')
print('  • event_distribution.png')
print('  • source_ip_activity.png')
print('  • temporal_distribution.png')
print('  • event_heatmap.png')
print('  • feature_distributions.png')
print('  • feature_boxplots.png')
print('  • correlation_heatmap.png')
print('  • missing_values.png (if applicable)')